In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

os.chdir('/content/drive/MyDrive/GNNs/HIV inhibitors-GNN')
print("Current directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content/drive/MyDrive/GNNs/HIV inhibitors-GNN


In [2]:
!pip install rdkit deepchem torch_geometric

In [3]:
import deepchem as dc
from rdkit import Chem

# Example SMILES
smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"  # Aspirin

# Featurizer (most commonly used for GNNs in recent DeepChem)
featurizer = dc.feat.MolGraphConvFeaturizer(
    use_edges=True,          # ← important if you want edge features
    use_chirality=False,
    use_partial_charge=False
)

# Featurize single molecule
mol = Chem.MolFromSmiles(smiles)
graph = featurizer.featurize(mol)         # returns GraphData

graph

wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


array([<deepchem.feat.graph_data.GraphData object at 0x794e6f30f860>],
      dtype=object)

In [4]:
dir(graph[0])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'edge_features',
 'edge_index',
 'node_features',
 'node_pos_features',
 'num_edge_features',
 'num_edges',
 'num_node_features',
 'num_nodes',
 'to_dgl_graph',
 'to_pyg_graph']

In [5]:
g = graph[0].to_pyg_graph()

In [6]:
import pandas as pd , tqdm, torch


In [9]:
!ls src/dataset/

dataset_InMem_DeepChem.py  dataset_new.py  __init__.py
dataset_InMem.py	   dataset.py	   __pycache__


In [10]:
from src.dataset.dataset_InMem_DeepChem import MoleculeInMemoryDataset_DC

ROOT = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN/data"

train_dataset = MoleculeInMemoryDataset_DC(root=ROOT, filename="HIV_train_val.csv")
test_dataset = MoleculeInMemoryDataset_DC(root=ROOT, filename="HIV_test.csv", test=True)

Processing...
Processing molecules:  15%|█▌        | 5555/37008 [00:52<03:25, 153.24it/s][14:38:12] WARNING: not removing hydrogen atom without neighbors
[14:38:12] WARNING: not removing hydrogen atom without neighbors
Processing molecules: 100%|██████████| 37008/37008 [05:46<00:00, 106.90it/s]


Saving 37008 graphs to /content/drive/MyDrive/GNNs/HIV inhibitors-GNN/data/processed/train_val_molecules_deepchem.pt


Done!
Processing...
Processing molecules: 100%|██████████| 4112/4112 [00:38<00:00, 107.85it/s]


Saving 4112 graphs to /content/drive/MyDrive/GNNs/HIV inhibitors-GNN/data/processed/test_molecules_deepchem.pt


Done!


In [11]:
data = pd.read_csv('data/raw/HIV_train_val.csv')
data[data["HIV_active"] == 1].head()

featurizer = dc.feat.MolGraphConvFeaturizer(
    use_edges=True,          # ← important if you want edge features
    use_chirality=False,
    use_partial_charge=False
)

data = data[:2]
print(f"Number of data items: {len(data)}")

out = []

for _, row in tqdm.tqdm(data.iterrows(), total=len(data)):
  smiles = row['smiles']
  mol = Chem.MolFromSmiles(smiles)
  if mol is not None:
    graph = featurizer.featurize(mol)[0]
    y = torch.tensor([row['HIV_active']])
    g = graph.to_pyg_graph()
    g.y = y
    print(g)
    out.append(g)

Number of data items: 2


100%|██████████| 2/2 [00:00<00:00, 99.87it/s]

Data(x=[29, 30], edge_index=[2, 64], edge_attr=[64, 11], y=[1])
Data(x=[18, 30], edge_index=[2, 40], edge_attr=[40, 11], y=[1])


In [12]:
for i in out:
  print(i)


Data(x=[29, 30], edge_index=[2, 64], edge_attr=[64, 11], y=[1])
Data(x=[18, 30], edge_index=[2, 40], edge_attr=[40, 11], y=[1])


In [13]:
for i in range(2):
  g = train_dataset[i]
  print(g)

Data(x=[29, 30], edge_index=[2, 64], edge_attr=[64, 11], y=[1])
Data(x=[18, 30], edge_index=[2, 40], edge_attr=[40, 11], y=[1])
